# options_3m_strat_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (102).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `options` |
| Trading style | `unknown` |
| Timeframe | `3m` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, compute_indicators, atr_vec, prepare_data, backtest_ticker_numba, run_backtest, calculate_metrics, rank_and_score, backtest_with_params, optimize_params, simple_optimize, calculate_metrics |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2020_25.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2021-07-01'
end_date = '2025-12-30'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT', 'SOLUSDT', 'XRPUSDT', 'BNBUSDT', 'ETHUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('3min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

In [ ]:
data.dropna()
data

In [ ]:
import pandas as pd
import numpy as np
from numba import njit
from tqdm import tqdm

def compute_indicators(
    df,
    lookback=12,
    pa_threshold=0.45,
    te_threshold=0.35,
    extreme_frac=0.25,
    atr_mult=0.5,
):
    """Calculate indicators, ATR-based trailing stop, and trading signals"""
    df = df.copy()
    o, h, l, c = df["open"].values, df["high"].values, df["low"].values, df["close"].values
    n = len(df)

    # Vectorized ATR calculation
    def atr_vec(period):
        tr1 = h - l
        tr2 = np.abs(h - np.roll(c, 1))
        tr3 = np.abs(l - np.roll(c, 1))
        tr = np.maximum(np.maximum(tr1, tr2), tr3)
        tr[0] = tr1[0]  # Fix first value
        return pd.Series(tr).rolling(period).mean().values

    atr_14 = atr_vec(14)
    atr_5 = atr_vec(5)

    atr_14_max = pd.Series(atr_14).rolling(20).max().values
    atr_5_max = pd.Series(atr_5).rolling(20).max().values

    df["atr_trail"] = np.maximum(atr_14_max, atr_5_max) * atr_mult
    df["atr_trail"] = np.where(np.isnan(df["atr_trail"]), 0.0001, df["atr_trail"])

    # Vectorized Path Asymmetry
    net_move = np.abs(c - np.roll(o, lookback))
    path_len = pd.Series(np.abs(np.diff(c, prepend=c[0]))).rolling(lookback).sum().values
    df["path_asym"] = np.where(path_len != 0, net_move / path_len, 0.0)
    df["path_asym"] = np.nan_to_num(df["path_asym"], 0.0)

    # Vectorized Time at Extremes
    hh = pd.Series(h).rolling(lookback).max().values
    ll = pd.Series(l).rolling(lookback).min().values
    eps = extreme_frac * (hh - ll)

    te_high = pd.Series(c > (hh - eps)).rolling(lookback).mean().values
    te_low = pd.Series(c < (ll + eps)).rolling(lookback).mean().values
    df["te_high"] = te_high
    df["te_low"] = te_low

    # Vectorized Direction
    direction = np.sign(c - np.roll(o, lookback))
    direction[np.isnan(direction)] = 0
    direction = direction.astype(int)

    # Vectorized Signal Generation
    long_cond = (direction > 0) & (df["path_asym"] >= pa_threshold) & (te_high >= te_threshold)
    short_cond = (direction < 0) & (df["path_asym"] >= pa_threshold) & (te_low >= te_threshold)

    df["signal"] = 0
    df.loc[long_cond, "signal"] = 1
    df.loc[short_cond, "signal"] = -1

    return df


def prepare_data(data):
    """Prepare data with indicators for all tickers"""
    data = data.copy()

    if 'Date' in data.columns:
        data['date'] = pd.to_datetime(data['Date'])
        data.drop(columns=['Date'], inplace=True)

    if 'Ticker' in data.columns:
        data.rename(columns={'Ticker': 'ticker'}, inplace=True)

    # Process all tickers at once using groupby.apply (faster than loop)
    data = data.sort_values(['ticker', 'date'])
    data = data.groupby('ticker', group_keys=False).apply(compute_indicators)

    # Shift to avoid look-ahead bias
    data['signal'] = data.groupby('ticker')['signal'].shift(1)
    data['atr_trail'] = data.groupby('ticker')['atr_trail'].shift(1)

    # Clean data
    data = data.dropna(subset=['signal', 'atr_trail'])
    data['signal'] = data['signal'].astype(int)

    return data[['date', 'ticker', 'open', 'high', 'low', 'close', 'signal', 'atr_trail']].reset_index(drop=True)


@njit
def backtest_ticker_numba(opens, highs, lows, closes, signals, atr_trails, capital,
                          fee_rate, slippage_rate, initial_sl_pct):
    """Numba-compiled backtest for maximum speed"""
    n = len(opens)

    # Pre-allocate arrays for results
    entry_idx = []
    exit_idx = []
    entry_prices = []
    exit_prices = []
    quantities = []
    directions = []
    pnls = []
    exit_reasons = []

    position_active = False
    position_direction = 0
    position_entry_price = 0.0
    position_size = 0.0
    trailing_stop = 0.0
    position_entry_idx = 0

    for i in range(n):
        if not position_active:
            if signals[i] != 0:
                direction = signals[i]
                entry_price = opens[i] * (1 + slippage_rate * direction)
                position_size = capital / entry_price

                if direction == 1:
                    initial_stop_loss = opens[i] * (1 - initial_sl_pct)
                else:
                    initial_stop_loss = opens[i] * (1 + initial_sl_pct)

                # Check initial SL on entry candle
                if direction == 1:
                    if lows[i] <= initial_stop_loss:
                        exit_price = initial_stop_loss * (1 - slippage_rate * direction)
                        gross_pnl = (exit_price - entry_price) * position_size
                        total_fees = fee_rate * (entry_price + exit_price) * position_size
                        net_pnl = gross_pnl - total_fees

                        entry_idx.append(i)
                        exit_idx.append(i)
                        entry_prices.append(entry_price)
                        exit_prices.append(exit_price)
                        quantities.append(position_size)
                        directions.append(1)
                        pnls.append(net_pnl)
                        exit_reasons.append(0)  # 0 = INITIAL_SL
                        continue
                else:
                    if highs[i] >= initial_stop_loss:
                        exit_price = initial_stop_loss * (1 - slippage_rate * direction)
                        gross_pnl = (entry_price - exit_price) * position_size
                        total_fees = fee_rate * (entry_price + exit_price) * position_size
                        net_pnl = gross_pnl - total_fees

                        entry_idx.append(i)
                        exit_idx.append(i)
                        entry_prices.append(entry_price)
                        exit_prices.append(exit_price)
                        quantities.append(position_size)
                        directions.append(-1)
                        pnls.append(net_pnl)
                        exit_reasons.append(0)
                        continue

                # Open position
                position_active = True
                position_entry_price = entry_price
                position_direction = direction
                trailing_stop = initial_stop_loss
                position_entry_idx = i

        else:
            # Position management
            previous_close = closes[i - 1]

            if position_direction == 1:
                new_trailing_stop = previous_close - atr_trails[i]
                trailing_stop = max(trailing_stop, new_trailing_stop)

                if lows[i] <= trailing_stop:
                    exit_price = trailing_stop * (1 - slippage_rate * position_direction)
                    gross_pnl = (exit_price - position_entry_price) * position_size
                    total_fees = fee_rate * (position_entry_price + exit_price) * position_size
                    net_pnl = gross_pnl - total_fees

                    entry_idx.append(position_entry_idx)
                    exit_idx.append(i)
                    entry_prices.append(position_entry_price)
                    exit_prices.append(exit_price)
                    quantities.append(position_size)
                    directions.append(1)
                    pnls.append(net_pnl)
                    exit_reasons.append(1)  # 1 = TRAILING_SL
                    position_active = False

            else:
                new_trailing_stop = previous_close + atr_trails[i]
                trailing_stop = min(trailing_stop, new_trailing_stop)

                if highs[i] >= trailing_stop:
                    exit_price = trailing_stop * (1 - slippage_rate * position_direction)
                    gross_pnl = (position_entry_price - exit_price) * position_size
                    total_fees = fee_rate * (position_entry_price + exit_price) * position_size
                    net_pnl = gross_pnl - total_fees

                    entry_idx.append(position_entry_idx)
                    exit_idx.append(i)
                    entry_prices.append(position_entry_price)
                    exit_prices.append(exit_price)
                    quantities.append(position_size)
                    directions.append(-1)
                    pnls.append(net_pnl)
                    exit_reasons.append(1)
                    position_active = False

    return (entry_idx, exit_idx, entry_prices, exit_prices,
            quantities, directions, pnls, exit_reasons)


def run_backtest(data, initial_capital=100000, fee_rate=0.00025,
                 slippage_rate=0.001, initial_sl_pct=0.02):
    """Run backtest across all tickers using Numba"""
    all_trades = []

    for ticker in tqdm(data['ticker'].unique(), desc="Backtesting"):
        ticker_data = data[data['ticker'] == ticker].sort_values('date').reset_index(drop=True)

        # Convert to numpy arrays for Numba
        opens = ticker_data['open'].values
        highs = ticker_data['high'].values
        lows = ticker_data['low'].values
        closes = ticker_data['close'].values
        signals = ticker_data['signal'].values
        atr_trails = ticker_data['atr_trail'].values
        dates = ticker_data['date'].values

        # Run Numba-compiled backtest
        results = backtest_ticker_numba(
            opens, highs, lows, closes, signals, atr_trails,
            initial_capital, fee_rate, slippage_rate, initial_sl_pct
        )

        entry_idx, exit_idx, entry_prices, exit_prices, quantities, directions, pnls, exit_reasons = results

        # Convert to trades
        for j in range(len(entry_idx)):
            all_trades.append({
                'ticker': ticker,
                'entry_date': dates[entry_idx[j]],
                'exit_date': dates[exit_idx[j]],
                'entry_price': entry_prices[j],
                'exit_price': exit_prices[j],
                'qty': quantities[j],
                'direction': 'LONG' if directions[j] == 1 else 'SHORT',
                'pnl': pnls[j],
                'exit_reason': 'INITIAL_SL' if exit_reasons[j] == 0 else 'TRAILING_SL'
            })

    if not all_trades:
        return pd.DataFrame()

    df_trades = pd.DataFrame(all_trades)
    df_trades = df_trades.sort_values(['ticker', 'entry_date']).reset_index(drop=True)
    df_trades['cumulative_pnl'] = df_trades.groupby('ticker')['pnl'].cumsum()

    # Format tradebook
    tradebook = pd.DataFrame({
        'Ticker': df_trades['ticker'],
        'Entry Time': df_trades['entry_date'],
        'Exit Time': df_trades['exit_date'],
        'Entry Price': df_trades['entry_price'],
        'Exit Price': df_trades['exit_price'],
        'PnL': df_trades['pnl'],
        'Cumulative PnL': df_trades['cumulative_pnl'],
        'Direction': df_trades['direction'],
        'Exit Reason': df_trades['exit_reason']
    })

    return tradebook


# Usage
data = prepare_data(data)
tradebook = run_backtest(data, initial_capital=100000, fee_rate=0.0003,
                        slippage_rate=0.0003, initial_sl_pct=0.002)
tradebook.to_csv("/content/drive/MyDrive/tradebook.csv", index=False)

In [ ]:
tradebook

In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
'''

def calculate_metrics(trades, capital):
    """Calculate performance metrics from trades"""
    if not trades:
        return None

    pnls = np.array([t['pnl'] for t in trades])
    pnl = pnls.sum()

    if pnl <= 0:  # Early exit for unprofitable
        return None

    wins = pnls[pnls > 0].sum()
    losses = np.abs(pnls[pnls < 0].sum())
    pf = wins / losses if losses > 0 else 0

    if pf < 1.0:  # Early exit for PF < 1
        return None

    cumsum = np.cumsum(pnls)
    dd_pct = ((cumsum - np.maximum.accumulate(cumsum)).min() / capital) * 100

    # Directional R-squared
    n = len(cumsum)
    if n > 1:
        x = np.arange(n)
        slope = np.cov(x, cumsum)[0, 1] / np.var(x)
        intercept = cumsum.mean() - slope * x.mean()
        y_pred = slope * x + intercept
        ss_res = np.sum((cumsum - y_pred) ** 2)
        ss_tot = np.sum((cumsum - cumsum.mean()) ** 2)
        r2_raw = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        r2 = r2_raw if slope > 0 else -r2_raw
    else:
        r2 = 0

    return {
        'pnl': pnl,
        'pf': pf,
        'dd': dd_pct,
        'r2': r2,
        'n_trades': n,
        'wr': (pnls > 0).sum() / n * 100
    }


def rank_and_score(results, weights={'pf': 0.25, 'dd': 0.25, 'r2': 0.25, 'n_trades': 0.15, 'wr': 0.10}):
    """Rank results and compute composite score"""
    if not results:
        return []

    metrics = np.array([[r['metrics'][k] for k in ['pf', 'r2', 'n_trades', 'wr', 'dd']]
                        for r in results])

    # Vectorized ranking
    ranks = np.zeros_like(metrics)
    for i in range(4):  # pf, r2, n_trades, wr
        if metrics[:, i].std() > 0:
            ranks[:, i] = (metrics[:, i].argsort().argsort() + 1) / len(metrics)
        else:
            ranks[:, i] = 0.5

    # DD (invert)
    if metrics[:, 4].std() > 0:
        ranks[:, 4] = 1 - (metrics[:, 4].argsort().argsort() + 1) / len(metrics)
    else:
        ranks[:, 4] = 0.5

    scores = (ranks[:, 0] * weights['pf'] +
              ranks[:, 1] * weights['r2'] +
              ranks[:, 2] * weights['n_trades'] +
              ranks[:, 3] * weights['wr'] +
              ranks[:, 4] * weights['dd'])

    for i, r in enumerate(results):
        r['score'] = scores[i]

    return results


def backtest_with_params(data, params, ticker, capital, fee, slip):
    """Backtest single parameter set using backtest_ticker_numba"""
    df = data[data['ticker'] == ticker].copy()
    if len(df) < 50:
        return None

    df = df.sort_values('date').reset_index(drop=True)

    # Extract indicator parameters (exclude initial_sl and mult)
    indicator_params = {k: v for k, v in params.items() if k not in ['initial_sl', 'mult']}
    df = compute_indicators(df, **indicator_params, atr_mult=params['mult'])

    # Shift to avoid look-ahead bias
    df['signal'] = df['signal'].shift(1)
    df['atr_trail'] = df['atr_trail'].shift(1)

    # Remove NaN rows
    df = df.dropna(subset=['signal', 'atr_trail'])

    if len(df) == 0:
        return None

    # Convert to numpy arrays
    opens = df['open'].values
    highs = df['high'].values
    lows = df['low'].values
    closes = df['close'].values
    signals = df['signal'].values
    atr_trails = df['atr_trail'].values
    dates = df['date'].values

    # Call backtest_ticker_numba directly
    results = backtest_ticker_numba(
        opens, highs, lows, closes, signals, atr_trails,
        capital, fee, slip, params['initial_sl']
    )

    entry_idx, exit_idx, entry_prices, exit_prices, quantities, directions, pnls, exit_reasons = results

    # Convert to trades list
    trades = []
    for j in range(len(entry_idx)):
        trades.append({
            'entry_date': dates[entry_idx[j]],
            'exit_date': dates[exit_idx[j]],
            'entry_price': entry_prices[j],
            'exit_price': exit_prices[j],
            'qty': quantities[j],
            'direction': 'LONG' if directions[j] == 1 else 'SHORT',
            'pnl': pnls[j],
            'exit_reason': 'INITIAL_SL' if exit_reasons[j] == 0 else 'TRAILING_SL'
        })

    return trades


def optimize_params(combo, keys, data, ticker, capital, fee, slip, min_trades):
    """Worker: test one param combo"""
    params = dict(zip(keys, combo))
    trades = backtest_with_params(data, params, ticker, capital, fee, slip)

    if not trades or len(trades) < min_trades:
        return None

    metrics = calculate_metrics(trades, capital)

    if metrics is None:
        return None

    return {'params': params, 'metrics': metrics, 'trades': trades}


def simple_optimize(data, param_grid, capital=100000, fee=0.00025, slip=0.001,
                    min_pf=1.0, min_trades=10):
    """
    Test all parameter combinations to find profitable strategies

    Returns: results_df, best_params, tradebook
    """

    print(f"\n{'='*80}")
    print(f"OPTIMIZATION STARTED")
    print(f"Period: {data['date'].min().date()} → {data['date'].max().date()}")
    print(f"Tickers: {len(data['ticker'].unique())}")
    print(f"Combinations: {np.prod([len(v) for v in param_grid.values()]):,.0f}")
    print(f"{'='*80}\n")

    keys = list(param_grid.keys())
    param_combos = list(product(*[param_grid[k] for k in keys]))
    all_results = []

    for ticker in data['ticker'].unique():
        print(f"\nProcessing {ticker}...")

        # Use partial to reduce argument passing overhead
        worker = partial(optimize_params,
                        keys=keys,
                        data=data,
                        ticker=ticker,
                        capital=capital,
                        fee=fee,
                        slip=slip,
                        min_trades=min_trades)

        # Use chunksize for better performance
        chunksize = max(1, len(param_combos) // (cpu_count() * 4))

        with Pool(cpu_count()) as pool:
            results = list(tqdm(
                pool.imap(worker, param_combos, chunksize=chunksize),
                total=len(param_combos),
                desc=f"{ticker}"
            ))

        results = [r for r in results if r is not None]

        if not results:
            print(f"  No profitable combinations for {ticker}")
            continue

        print(f"  Found {len(results)} profitable combinations")
        results = rank_and_score(results)

        for r in results:
            r['ticker'] = ticker
        all_results.extend(results)

    if not all_results:
        print("\n⚠️  NO PROFITABLE COMBINATIONS FOUND")
        return pd.DataFrame(), {}, pd.DataFrame()

    # Create results DataFrame
    results_df = pd.DataFrame([{
        'ticker': r['ticker'],
        'score': r['score'],
        **{f'param_{k}': v for k, v in r['params'].items()},
        **r['metrics']
    } for r in all_results])

    results_df = results_df.sort_values('score', ascending=False).reset_index(drop=True)

    # Best result
    best_result = max(all_results, key=lambda x: x['score'])
    best_params = best_result['params']

    print(f"\n{'='*80}")
    print(f"BEST RESULT - {best_result['ticker']}")
    print(f"Score: {best_result['score']:.3f}")
    print(f"Parameters: {best_params}")
    m = best_result['metrics']
    print(f"PnL: ${m['pnl']:,.0f} | PF: {m['pf']:.2f} | DD: {m['dd']:.1f}%")
    print(f"Trades: {m['n_trades']} | WR: {m['wr']:.1f}% | R²: {m['r2']:.3f}")
    print(f"{'='*80}\n")

    # Generate tradebook
    tradebook = pd.DataFrame(best_result['trades'])
    if not tradebook.empty:
        tradebook = tradebook.sort_values('entry_date').reset_index(drop=True)

        # Format tradebook for export
        output = pd.DataFrame({
            'Entry Time': tradebook['entry_date'],
            'Exit Time': tradebook['exit_date'],
            'Ticker': best_result['ticker'],
            'Direction': tradebook['direction'],
            'Entry Price': tradebook['entry_price'].round(2),
            'Exit Price': tradebook['exit_price'].round(2),
            'PnL': tradebook['pnl'].round(2),
            'Exit Reason': tradebook['exit_reason']
        })

        # Save files
        results_df.to_csv('optimization_results.csv', index=False)
        output.to_csv('best_tradebook.csv', index=False)

        print(f"✓ Saved: optimization_results.csv ({len(results_df)} combinations)")
        print(f"✓ Saved: best_tradebook.csv ({len(output)} trades)\n")

    return results_df, best_params, tradebook


# ============================================================================
# USAGE
# ============================================================================

param_grid = {
    'lookback': [12, 18],
    'pa_threshold': [0.4, 0.45, 0.5],
    'te_threshold': [0.3, 0.35, 0.45],
    'extreme_frac': [0.15, 0.2, 0.25, 0.3, 0.35],
    'mult': [0.05, 0.1, 0.2, 0.5],
    'initial_sl': [0.01, 0.015, 0.02]
}

results_df, best_params, tradebook = simple_optimize(
    data=data,
    param_grid=param_grid,
    capital=100000,
    fee=0.0003,
    slip=0.0003,
    min_pf=1.0,
    min_trades=10
)

print("\nTop 10 Results:")
print(results_df.head(10)[['ticker', 'score', 'pnl', 'pf', 'dd', 'r2', 'n_trades', 'wr']]) '''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
'''
# Load optimization results
df = pd.read_csv('optimization_results.csv')

# Extract parameter columns
param_cols = [col for col in df.columns if col.startswith('param_')]
metric_cols = ['pnl', 'pf', 'dd', 'r2', 'n_trades', 'wr', 'score']

# Create a comprehensive visualization
fig, axes = plt.subplots(len(param_cols), len(metric_cols), figsize=(24, 4*len(param_cols)))
fig.suptitle('Parameter Sensitivity Analysis - Metrics vs Parameters', fontsize=16, y=1.0)

for i, param in enumerate(param_cols):
    param_name = param.replace('param_', '')

    for j, metric in enumerate(metric_cols):
        ax = axes[i, j] if len(param_cols) > 1 else axes[j]

        # Group by parameter and calculate statistics
        grouped = df.groupby(param)[metric].agg(['mean', 'std', 'min', 'max'])

        # Plot mean with error bars
        x = grouped.index
        y_mean = grouped['mean']
        y_std = grouped['std']

        ax.plot(x, y_mean, 'o-', linewidth=2, markersize=8, label='Mean')
        ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.3)

        # Add min/max range
        ax.fill_between(x, grouped['min'], grouped['max'], alpha=0.1, color='gray')

        # Styling
        ax.set_xlabel(param_name, fontsize=10, fontweight='bold')
        ax.set_ylabel(metric.upper(), fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_title(f'{metric.upper()} vs {param_name}', fontsize=9)

        # Highlight plateaus (low std relative to mean)
        if len(grouped) > 2 and y_mean.std() > 0:
            stability = y_std / (y_mean.std() + 1e-10)
            stable_regions = stability < stability.median()
            if stable_regions.any():
                ax.axvspan(x[stable_regions].min(), x[stable_regions].max(),
                          alpha=0.1, color='green', label='Stable Region')

plt.tight_layout()
plt.savefig('parameter_sensitivity_full.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Saved: parameter_sensitivity_full.png")

# ============================================================================
# Individual Parameter Plots (More Detailed)
# ============================================================================

for param in param_cols:
    param_name = param.replace('param_', '')

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle(f'Detailed Analysis: {param_name.upper()}', fontsize=16, fontweight='bold')

    for idx, metric in enumerate(metric_cols):
        ax = axes[idx // 4, idx % 4]

        # Box plot for distribution
        param_values = df[param].unique()
        data_to_plot = [df[df[param] == val][metric].values for val in sorted(param_values)]

        bp = ax.boxplot(data_to_plot, labels=sorted(param_values), patch_artist=True)

        # Color boxes by median value
        medians = [np.median(d) for d in data_to_plot]
        colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(medians)))
        sorted_indices = np.argsort(medians)

        for patch, idx in zip(bp['boxes'], sorted_indices):
            patch.set_facecolor(colors[np.where(sorted_indices == list(bp['boxes']).index(patch))[0][0]])

        # Add mean line
        means = [np.mean(d) for d in data_to_plot]
        ax.plot(range(1, len(means) + 1), means, 'D-', color='red',
                linewidth=2, markersize=8, label='Mean', zorder=10)

        ax.set_xlabel(param_name, fontsize=11, fontweight='bold')
        ax.set_ylabel(metric.upper(), fontsize=11, fontweight='bold')
        ax.set_title(f'{metric.upper()} Distribution', fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        ax.legend()

        # Rotate x labels if needed
        if len(param_values) > 5:
            ax.tick_params(axis='x', rotation=45)

    # Remove empty subplot
    if len(metric_cols) < 8:
        fig.delaxes(axes[1, 3])

    plt.tight_layout()
    plt.savefig(f'parameter_detail_{param_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Saved: parameter_detail_{param_name}.png")

# ============================================================================
# Heatmap: Parameter Interactions
# ============================================================================

print("\n" + "="*80)
print("PARAMETER INTERACTION ANALYSIS")
print("="*80)

for i, param1 in enumerate(param_cols[:-1]):
    for param2 in param_cols[i+1:]:
        param1_name = param1.replace('param_', '')
        param2_name = param2.replace('param_', '')

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f'Interaction: {param1_name.upper()} vs {param2_name.upper()}',
                     fontsize=16, fontweight='bold')

        for idx, metric in enumerate(['pnl', 'pf', 'dd', 'r2', 'score', 'wr']):
            ax = axes[idx // 3, idx % 3]

            # Create pivot table
            pivot = df.pivot_table(values=metric, index=param1, columns=param2, aggfunc='mean')

            # Plot heatmap
            im = ax.imshow(pivot, cmap='RdYlGn', aspect='auto')

            # Set ticks
            ax.set_xticks(np.arange(len(pivot.columns)))
            ax.set_yticks(np.arange(len(pivot.index)))
            ax.set_xticklabels(pivot.columns)
            ax.set_yticklabels(pivot.index)

            # Add colorbar
            cbar = plt.colorbar(im, ax=ax)
            cbar.set_label(metric.upper(), rotation=270, labelpad=20)

            # Add values in cells
            for i in range(len(pivot.index)):
                for j in range(len(pivot.columns)):
                    if not np.isnan(pivot.iloc[i, j]):
                        text = ax.text(j, i, f'{pivot.iloc[i, j]:.2f}',
                                     ha="center", va="center", color="black", fontsize=8)

            ax.set_xlabel(param2_name, fontsize=10, fontweight='bold')
            ax.set_ylabel(param1_name, fontsize=10, fontweight='bold')
            ax.set_title(f'{metric.upper()}', fontsize=10)

        plt.tight_layout()
        plt.savefig(f'interaction_{param1_name}_vs_{param2_name}.png', dpi=150, bbox_inches='tight')
        plt.show()

        print(f"Saved: interaction_{param1_name}_vs_{param2_name}.png")

# ============================================================================
# Statistical Summary
# ============================================================================

print("\n" + "="*80)
print("PARAMETER STABILITY ANALYSIS")
print("="*80)

for param in param_cols:
    param_name = param.replace('param_', '')
    print(f"\n{param_name.upper()}:")
    print("-" * 60)

    for metric in ['score', 'pnl', 'pf', 'r2']:
        grouped = df.groupby(param)[metric].agg(['mean', 'std', 'count'])

        # Calculate coefficient of variation (lower = more stable)
        grouped['cv'] = grouped['std'] / (grouped['mean'].abs() + 1e-10)

        # Find most stable parameter value
        most_stable = grouped['cv'].idxmin()
        highest_mean = grouped['mean'].idxmax()

        print(f"  {metric.upper()}:")
        print(f"    Most Stable Value: {most_stable} (CV={grouped.loc[most_stable, 'cv']:.3f})")
        print(f"    Highest Mean Value: {highest_mean} (Mean={grouped.loc[highest_mean, 'mean']:.2f})")

# ============================================================================
# Best Parameter Combinations
# ============================================================================

print("\n" + "="*80)
print("TOP 10 PARAMETER COMBINATIONS BY SCORE")
print("="*80)

top_10 = df.nlargest(10, 'score')[param_cols + metric_cols]
print(top_10.to_string(index=False))

# ============================================================================
# Parameter Frequency in Top Results
# ============================================================================

print("\n" + "="*80)
print("PARAMETER FREQUENCY IN TOP 20% OF RESULTS")
print("="*80)

top_20_pct = df.nlargest(int(len(df) * 0.2), 'score')

for param in param_cols:
    param_name = param.replace('param_', '')
    freq = top_20_pct[param].value_counts(normalize=True).sort_index()

    print(f"\n{param_name.upper()}:")
    for val, pct in freq.items():
        print(f"  {val}: {pct*100:.1f}%")

print("\n" + "="*80)
print("VISUALIZATION COMPLETE")
print("="*80)'''

In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from multiprocessing import Pool, cpu_count
from tqdm import tqdm


# ============================================================================
# CORE METRICS
# ============================================================================

def calculate_metrics(trades, capital):
    """Calculate performance metrics from trades"""
    if not trades:
        return {'pnl': 0, 'pf': 0, 'dd': 0, 'r2': 0, 'n_trades': 0, 'wr': 0}

    df = pd.DataFrame(trades)
    pnl = df['pnl'].sum()
    wins = df[df['pnl'] > 0]['pnl'].sum()
    losses = abs(df[df['pnl'] < 0]['pnl'].sum())
    cumsum = df['pnl'].cumsum()
    dd_pct = ((cumsum - cumsum.cummax()).min() / capital) * 100

    # Directional R² (negative if downward sloping)
    equity = cumsum.values
    x = np.arange(len(equity))
    if len(x) > 1:
        slope, intercept = np.polyfit(x, equity, 1)
        y_pred = slope * x + intercept
        ss_res = np.sum((equity - y_pred) ** 2)
        ss_tot = np.sum((equity - np.mean(equity)) ** 2)
        r2_raw = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
        r2 = r2_raw if slope > 0 else -r2_raw
    else:
        r2 = 0

    return {
        'pnl': pnl,
        'pf': wins / losses if losses > 0 else 0,
        'dd': dd_pct,
        'r2': r2,
        'n_trades': len(df),
        'wr': (df['pnl'] > 0).sum() / len(df) * 100
    }


def rank_and_score(results, weights={'pf': 0.25, 'dd': 0.25, 'r2': 0.25, 'n_trades': 0.15, 'wr': 0.10}):
    """Rank results and compute composite score"""
    if not results:
        return []

    metrics = pd.DataFrame([r['metrics'] for r in results])
    ranks = pd.DataFrame({
        col: metrics[col].rank(pct=True) if metrics[col].std() > 0 else 0.5
        for col in ['pf', 'r2', 'n_trades', 'wr']
    })
    ranks['dd'] = 1 - metrics['dd'].rank(pct=True) if metrics['dd'].std() > 0 else 0.5
    scores = sum(ranks[k] * weights[k] for k in weights)

    for i, r in enumerate(results):
        r['score'] = scores.iloc[i]

    return results


# ============================================================================
# BACKTEST EXECUTION
# ============================================================================

def backtest_params(data, params, ticker, capital, fee, slip):
    """Backtest single parameter set"""
    df = data[data['ticker'] == ticker].copy()
    if len(df) < 50:
        return None

    df = df.sort_values('date').reset_index(drop=True)

    # Compute indicators with params
    df_calc = compute_indicators(
        df,
        lookback=params['lookback'],
        pa_threshold=params['pa_threshold'],
        te_threshold=params['te_threshold'],
        extreme_frac=params['extreme_frac'],
        atr_mult=params['atr_mult']
    )

    # Shift to avoid look-ahead
    df_calc['signal'] = df_calc['signal'].shift(1)
    df_calc['atr_trail'] = df_calc['atr_trail'].shift(1)
    df_calc = df_calc.dropna(subset=['signal', 'atr_trail'])

    if len(df_calc) == 0:
        return None

    # Run backtest
    results = backtest_ticker_numba(
        df_calc['open'].values,
        df_calc['high'].values,
        df_calc['low'].values,
        df_calc['close'].values,
        df_calc['signal'].values.astype(int),
        df_calc['atr_trail'].values,
        capital, fee, slip, params['initial_sl']
    )

    entry_idx, exit_idx, entry_prices, exit_prices, quantities, directions, pnls, exit_reasons = results

    # Convert to trades
    trades = []
    for j in range(len(entry_idx)):
        trades.append({
            'entry_date': df_calc['date'].values[entry_idx[j]],
            'exit_date': df_calc['date'].values[exit_idx[j]],
            'entry_price': entry_prices[j],
            'exit_price': exit_prices[j],
            'qty': quantities[j],
            'direction': 'LONG' if directions[j] == 1 else 'SHORT',
            'pnl': pnls[j],
            'exit_reason': 'INITIAL_SL' if exit_reasons[j] == 0 else 'TRAILING_SL'
        })

    return trades if trades else None


def optimize_worker(args):
    """Worker: test one param combo"""
    combo, keys, data, ticker, capital, fee, slip, min_pf, min_trades = args
    params = dict(zip(keys, combo))
    trades = backtest_params(data, params, ticker, capital, fee, slip)

    if not trades or len(trades) < min_trades:
        return None

    metrics = calculate_metrics(trades, capital)
    if metrics['pnl'] <= 0 or metrics['pf'] < min_pf:
        return None

    return {'params': params, 'metrics': metrics}


# ============================================================================
# PARAMETER SELECTION
# ============================================================================

def select_best_params(data, param_combos, keys, ticker, capital, fee, slip,
                      min_pf, min_trades, method, param_grid):
    """Select ONE robust parameter set from training data"""

    print(f"\n{'='*60}")
    print(f"PARAMETER SELECTION: {ticker} ({method.upper()})")
    print(f"{'='*60}")

    # Test all parameter combinations
    args = [(c, keys, data, ticker, capital, fee, slip, min_pf, min_trades) for c in param_combos]

    with Pool(cpu_count()) as pool:
        results = []
        for r in tqdm(pool.imap(optimize_worker, args),
                     total=len(args), desc=f"Testing {ticker}", ncols=80):
            if r is not None:
                results.append(r)

    if not results:
        print(f"⚠️  No profitable params found")
        return None

    print(f"✓ Found {len(results)} profitable sets")
    results = rank_and_score(results)

    # Select based on method
    if method == 'best':
        selected = max(results, key=lambda x: x['score'])['params']

    elif method == 'median_top20':
        n_top = max(1, int(len(results) * 0.2))
        top = sorted(results, key=lambda x: x['score'], reverse=True)[:n_top]

        # Handle single result case
        if len(top) == 1:
            selected = top[0]['params']
        else:
            param_df = pd.DataFrame([r['params'] for r in top])

            # Get median and find nearest value from original grid
            selected = {}
            for k in keys:
                median_val = param_df[k].median()
                # Find closest value in the original parameter grid
                grid_values = param_grid[k]
                selected[k] = min(grid_values, key=lambda x: abs(x - median_val))

    elif method == 'robust':
        n_top = max(1, int(len(results) * 0.1))
        top = sorted(results, key=lambda x: x['score'], reverse=True)[:n_top]

        # Handle single result case
        if len(top) == 1:
            selected = top[0]['params']
        else:
            counts = {k: {} for k in keys}
            for r in top:
                for k, v in r['params'].items():
                    counts[k][v] = counts[k].get(v, 0) + 1
            selected = {k: max(v.items(), key=lambda x: x[1])[0] for k, v in counts.items()}

    # Print selection
    for k, v in selected.items():
        print(f"  {k}: {v}")

    # Show training performance
    match = next((r for r in results if r['params'] == selected), None)
    if match:
        m = match['metrics']
        print(f"\nTraining: PnL ${m['pnl']:,.0f} | PF {m['pf']:.2f} | "
              f"R² {m['r2']:.3f} | Trades {m['n_trades']}")

    return selected


# ============================================================================
# WALK-FORWARD ENGINE
# ============================================================================

def create_folds(data, train_months, test_months, anchored=False):
    """Generate time-based train/test folds"""
    data = data.sort_values('date')
    min_date, max_date = data['date'].min(), data['date'].max()
    folds, idx = [], 0
    anchor = min_date if anchored else None

    while True:
        fs = anchor if anchored else min_date + pd.DateOffset(months=idx * test_months)
        te = (min_date if anchored else fs) + pd.DateOffset(months=train_months + (idx * test_months if anchored else 0))
        fe = te + pd.DateOffset(months=test_months)

        if te > max_date or fe > max_date:
            break

        train = data[(data['date'] >= fs) & (data['date'] < te)]
        test = data[(data['date'] >= te) & (data['date'] < fe)]

        if len(train) >= 100 and len(test) >= 20:
            folds.append({'idx': idx, 'train': train, 'test': test,
                         'fs': fs, 'te': te, 'fe': fe})
        idx += 1

    return folds


def test_folds(folds, params, ticker, capital, fee, slip):
    """Apply fixed params to all test folds"""
    results = []
    all_trades = []

    for fold in tqdm(folds, desc=f"Testing {ticker}", ncols=80):
        trades = backtest_params(fold['test'], params, ticker, capital, fee, slip)

        if not trades:
            continue

        metrics = calculate_metrics(trades, capital)
        tqdm.write(f"Fold {fold['idx']}: ${metrics['pnl']:,.0f} | "
                  f"PF {metrics['pf']:.2f} | {metrics['n_trades']} trades")

        # Tag trades
        for t in trades:
            t.update({'fold': fold['idx'], 'ticker': ticker,
                     'params': str(params)})

        results.append({
            'fold': fold['idx'],
            'params': params,
            'metrics': metrics,
            'period': (fold['te'], fold['fe'])
        })

        all_trades.extend(trades)

    return results, all_trades


def walk_forward_optimize(data, param_grid,
                         train_months=12, test_months=3,
                         capital=100000, fee=0.0003, slip=0.0003,
                         anchored=False, min_pf=1.2, min_trades=15,
                         method='median_top20'):
    """
    Walk-Forward Optimization with FIXED parameters

    The CORRECT approach:
    1. Select ONE parameter set from initial training data
    2. Apply SAME parameters to ALL future test periods
    3. No re-optimization between folds

    Parameters:
    -----------
    method : str
        'median_top20' : Median of top 20% (RECOMMENDED - most robust)
        'best'         : Best single performer
        'robust'       : Most frequent in top 10%
    """

    print(f"\n{'='*80}")
    print(f"WALK-FORWARD OPTIMIZATION (FIXED PARAMS)")
    print(f"{'='*80}")
    print(f"Data: {data['date'].min().date()} → {data['date'].max().date()}")
    print(f"Tickers: {len(data['ticker'].unique())} | "
          f"Train: {train_months}m | Test: {test_months}m")
    print(f"Method: {method.upper()} | Mode: {'ANCHORED' if anchored else 'ROLLING'}")
    print(f"Filters: PF>{min_pf}, Trades≥{min_trades}")

    # Setup
    folds = create_folds(data, train_months, test_months, anchored)
    print(f"Folds: {len(folds)}\n")

    keys = list(param_grid.keys())
    combos = list(product(*[param_grid[k] for k in keys]))
    print(f"Testing {len(combos)} parameter combinations\n")

    ticker_results = {}
    all_trades = []

    # Process each ticker
    for ticker in tqdm(data['ticker'].unique(), desc="Tickers", ncols=80):
        tqdm.write(f"\n{'='*80}\nTICKER: {ticker}\n{'='*80}")

        # Select params from first training period only
        params = select_best_params(
            folds[0]['train'], combos, keys, ticker,
            capital, fee, slip, min_pf, min_trades, method, param_grid
        )

        if not params:
            tqdm.write(f"⚠️  Skipping {ticker}")
            continue

        # Apply same params to all test folds
        fold_results, trades = test_folds(folds, params, ticker, capital, fee, slip)

        if not fold_results:
            tqdm.write(f"⚠️  No test results for {ticker}")
            continue

        # Save fold summary
        summary = pd.DataFrame([{
            'fold': r['fold'],
            'test_start': r['period'][0].date(),
            'test_end': r['period'][1].date(),
            **{f'param_{k}': v for k, v in r['params'].items()},
            **{f'test_{k}': v for k, v in r['metrics'].items()}
        } for r in fold_results])

        summary.to_csv(f"wfo_{ticker}.csv", index=False)

        # Summary stats
        total_pnl = sum(r['metrics']['pnl'] for r in fold_results)
        total_trades = sum(r['metrics']['n_trades'] for r in fold_results)
        avg_r2 = np.mean([r['metrics']['r2'] for r in fold_results])

        tqdm.write(f"\n{'─'*60}")
        tqdm.write(f"SUMMARY: {ticker}")
        tqdm.write(f"{'─'*60}")
        tqdm.write(f"Fixed Params: {params}")
        tqdm.write(f"OOS PnL: ${total_pnl:,.2f} | Trades: {total_trades} | R²: {avg_r2:.3f}")
        tqdm.write(f"✓ Saved: wfo_{ticker}.csv")

        ticker_results[ticker] = {
            'params': params,
            'results': fold_results,
            'summary': summary,
            'pnl': total_pnl,
            'trades': total_trades
        }

        all_trades.extend(trades)

    # Generate tradebook
    print(f"\n{'='*80}\nTRADEBOOK\n{'='*80}")

    if all_trades:
        tb = pd.DataFrame(all_trades).sort_values(['ticker', 'entry_date'])

        output = pd.DataFrame({
            'Entry': tb['entry_date'],
            'Exit': tb['exit_date'],
            'Ticker': tb['ticker'],
            'Side': tb['direction'],
            'Entry Px': tb['entry_price'].round(2),
            'Exit Px': tb['exit_price'].round(2),
            'PnL': tb['pnl'].round(2),
            'Exit': tb['exit_reason'],
            'Fold': tb['fold'],
            'Params': tb['params']
        })

        output.to_csv("wfo_tradebook.csv", index=False)

        equity = output['PnL'].cumsum()
        dd = (equity - equity.cummax()).min()

        print(f"\nOUT-OF-SAMPLE PERFORMANCE:")
        print(f"  Trades: {len(output)} | PnL: ${output['PnL'].sum():,.2f}")
        print(f"  Win Rate: {(output['PnL']>0).mean()*100:.1f}%")
        print(f"  Avg: ${output['PnL'].mean():.2f} | Max DD: ${dd:,.2f}")
        print(f"  Final: ${capital + output['PnL'].sum():,.2f}")
        print(f"✓ Saved: wfo_tradebook.csv")
    else:
        output = pd.DataFrame()
        print("⚠️  No trades generated")

    print(f"{'='*80}\n")

    return ticker_results, output


# ============================================================================
# USAGE
# ============================================================================

if __name__ == '__main__':
    param_grid = {
        'lookback': [6, 12, 18],
        'pa_threshold': [0.4, 0.45, 0.5],
        'te_threshold': [0.3, 0.35, 0.4],
        'extreme_frac': [0.15, 0.25, 0.3, 0.35],
        'atr_mult': [0.1, 0.25, 0.5],
        'initial_sl': [0.01, 0.015, 0.02]
    }

    # Load your data: columns = ['date', 'ticker', 'open', 'high', 'low', 'close']
    # data = pd.read_csv('your_data.csv')
    # data['date'] = pd.to_datetime(data['date'])

    results, tradebook = walk_forward_optimize(
        data=data,
        param_grid=param_grid,
        train_months=1,      # Long training for robust param selection
        test_months=1,        # Test on 3-month windows
        capital=100000,
        fee=0.0003,
        slip=0.0003,
        anchored=False,       # Rolling window
        min_pf=1.0,          # Minimum profit factor
        min_trades=15,       # Minimum trades required
        method='median_top20' # Most robust selection method
    )